# Gold Layer — Daily Transaction Aggregation

**Objective:** Aggregate validated Silver transactions into daily financial KPIs by transaction date and type, and persist the results in a managed Delta Lake table using Databricks and PySpark.

### Why?

**For our project:** Provides reliable transaction volumes, status distributions, and completed financial amounts for operational reporting.

**For Data Engineering:** Demonstrates business-oriented data aggregation, conditional calculations, data consistency checks, and Delta Lake persistence within the Medallion architecture.

#### 01 — Read and Prepare Silver Data

Read the validated Silver transactions and derive `transaction_date` from `transaction_ts`. This establishes the daily aggregation key while preserving the original transaction data.

In [0]:
# Read validated transactions from Silver
df_silver = spark.table("workspace.default.fintech_transactions_silver")  
display(df_silver)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED


In [0]:
# Validate the initial Silver input count
assert df_silver.count() == 8, "Unexpected Silver row count"

In [0]:
from pyspark.sql.functions import col, to_date

# Extract the transaction date from the transaction timestamp
df_gold_prepared = df_silver.withColumn( "transaction_date", to_date(col("transaction_ts")) )
display(df_gold_prepared)

transaction_id,customer_id,transaction_ts,transaction_type,amount,status,transaction_date
TX007,C005,2026-10-18T13:20:00.000Z,PIX_IN,80.00,COMPLETED,2026-10-18
TX002,C001,2026-10-18T09:30:00.000Z,PIX_OUT,70.00,COMPLETED,2026-10-18
TX003,C002,2026-10-18T10:00:00.000Z,CARD_PURCHASE,90.50,COMPLETED,2026-10-18
TX004,C003,2026-10-18T11:00:00.000Z,BILL_PAYMENT,130.00,COMPLETED,2026-10-18
TX005,C004,2026-10-18T12:00:00.000Z,PIX_IN,400.00,PENDING,2026-10-18
TX006,C003,2026-10-18T13:00:00.000Z,PIX_OUT,120.00,FAILED,2026-10-18
TX008,C006,2026-10-18T14:00:00.000Z,PIX_OUT,35.00,COMPLETED,2026-10-18
TX001,C001,2026-10-18T09:00:00.000Z,PIX_IN,250.00,COMPLETED,2026-10-18


In [0]:
# Verify the derived date column and data types
df_gold_prepared.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- transaction_ts: timestamp (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- status: string (nullable = true)
 |-- transaction_date: date (nullable = true)



In [0]:
# Confirm that date extraction preserves all transactions
assert df_gold_prepared.count() == 8, "Unexpected Gold row count"

#### 02 — Calculate Daily Transaction KPIs

Aggregate transactions by date and transaction type using PySpark `groupBy()` and conditional aggregations. Calculate total transactions, completed, pending, and failed counts, and the total amount of completed transactions.

These KPIs provide daily operational metrics while excluding pending and failed amounts from completed financial totals.

In [0]:

import pyspark.sql.functions as sf

# Aggregate daily transactions and calculate status-based KPIs
df_gold_aggregated = df_gold_prepared.groupBy("transaction_date", "transaction_type").agg(
     sf.count("*").alias("total_transactions"),
     sf.sum(sf.when(col("status") == "COMPLETED", 1).otherwise(0)).alias("completed_transactions"),
     sf.sum(sf.when(col("status") == "PENDING", 1).otherwise(0)).alias("pending_transactions"),
     sf.sum(sf.when(col("status") == "FAILED", 1).otherwise(0)).alias("failed_transactions"),
     sf.sum(sf.when(col("status") == "COMPLETED", sf.col("amount")).otherwise(0)).alias("completed_amount")
)
display(df_gold_aggregated)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-18,PIX_IN,3,2,1,0,330.00
2026-10-18,PIX_OUT,3,2,0,1,105.00
2026-10-18,CARD_PURCHASE,1,1,0,0,90.50
2026-10-18,BILL_PAYMENT,1,1,0,0,130.00


#### 03 — Validate Gold Aggregations

Validate the expected number of Gold groups, transaction totals, completed amounts, and status consistency. These checks help detect incorrect aggregations before data persistence.

In [0]:
# Verify the expected number of daily transaction groups
assert df_gold_aggregated.count() == 4, "Unexpected Gold aggregated table row count"

In [0]:
import pyspark.sql.functions as sf

# Reconcile aggregated transaction counts with Silver
assert (
    df_gold_aggregated.select(sf.sum("total_transactions")).collect()[0][0] == 8
), "Unexpected total transaction in Gold aggregated table"


In [0]:
import pyspark.sql.functions as sf

# Validate the completed transaction amount
assert (
    df_gold_aggregated.select(sf.sum("completed_amount")).collect()[0][0] == 655.50
), "Unexpected total completed amount in Gold aggregated table"


In [0]:
import pyspark.sql.functions as sf

# Ensure status counts match total transactions in every group
status_total = (
    sf.col("completed_transactions")
    + sf.col("pending_transactions")
    + sf.col("failed_transactions")
)

df_inconsistent = df_gold_aggregated.filter(
    status_total != sf.col("total_transactions")
)

assert df_inconsistent.count() == 0, "Unexpected inconsistent status in Gold aggregated table"

#### 04 — Persist and Verify Gold Data

Persist the validated daily KPIs in a managed Delta Lake table. Read the stored records and verify their count, schema, and Delta transaction history to confirm successful persistence and data traceability.

In [0]:
# Create the initial Gold Delta table without replacing existing data
df_gold_aggregated.write.format("delta").mode("error").saveAsTable("workspace.default.fintech_transactions_daily_gold")

In [0]:
# Read the persisted Gold table to inspect the stored KPIs
df_gold_persisted = spark.table("workspace.default.fintech_transactions_daily_gold")  
display(df_gold_persisted)

transaction_date,transaction_type,total_transactions,completed_transactions,pending_transactions,failed_transactions,completed_amount
2026-10-18,CARD_PURCHASE,1,1,0,0,90.50
2026-10-18,PIX_IN,3,2,1,0,330.00
2026-10-18,PIX_OUT,3,2,0,1,105.00
2026-10-18,BILL_PAYMENT,1,1,0,0,130.00


In [0]:
# Verify the initial Gold table contains four groups
assert df_gold_persisted.count() == 4, \
    "Unexpected persisted Gold row count"

In [0]:
# Verify the persisted Gold schema and KPI data types
df_gold_persisted.printSchema()

root
 |-- transaction_date: date (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- completed_transactions: long (nullable = true)
 |-- pending_transactions: long (nullable = true)
 |-- failed_transactions: long (nullable = true)
 |-- completed_amount: decimal(22,2) (nullable = true)



In [0]:
%sql
-- Inspect Delta transaction history to verify table creation and data traceability
DESCRIBE HISTORY workspace.default.fintech_transactions_daily_gold;